# N03: PyTorch Full Pipeline - Target Encoding Inference

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
import os
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:

# 1. Load Full Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
sub_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


In [ ]:

# 2. Feature Engineering & Preprocessing
def engineer_features(df):
    df_new = df.copy()
    df_new['sleep_bin'] = pd.qcut(df_new['sleep_duration'], q=5, duplicates='drop').astype(str)
    df_new['stress_sleep_interact'] = df_new['stress_level'].astype(str) + '_' + df_new['sleep_bin']
    return df_new

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

# Target Encoding setup
X_train_cat = train_df[cat_cols].fillna('Missing').astype(str)
X_test_cat = test_df[cat_cols].fillna('Missing').astype(str)

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df['health_condition'])
class_names = le_target.classes_

# Missing Value Imputation and Scaling for Numeric
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_num = scaler.fit_transform(num_imputer.fit_transform(train_df[num_cols]))
X_test_num = scaler.transform(num_imputer.transform(test_df[num_cols]))


In [ ]:

# 3. K-Fold Cross-Fitted Target Encoding
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize arrays to hold target encoded features
num_classes = len(class_names)
X_train_te = np.zeros((len(train_df), len(cat_cols) * num_classes))
X_test_te = np.zeros((len(test_df), len(cat_cols) * num_classes))

for train_idx, val_idx in skf.split(X_train_cat, y_train):
    X_tr_fold, X_va_fold = X_train_cat.iloc[train_idx], X_train_cat.iloc[val_idx]
    y_tr_fold = y_train[train_idx]
    
    fold_te_train = []
    fold_te_val = []
    fold_te_test = []
    
    for i, col in enumerate(cat_cols):
        # Calculate probabilities per category on train fold
        crosstab = pd.crosstab(X_tr_fold[col], y_tr_fold, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        # Map to validation fold
        val_mapped = X_va_fold[col].map(mapping)
        val_mapped = val_mapped.apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(val_mapped.tolist()).values)
        
        # Map to test set (will average over folds)
        test_mapped = X_test_cat[col].map(mapping)
        test_mapped = test_mapped.apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(test_mapped.tolist()).values)
        
    X_train_te[val_idx] = np.hstack(fold_te_val)
    X_test_te += np.hstack(fold_te_test) / n_splits

te_scaler = StandardScaler()
X_train_te = te_scaler.fit_transform(X_train_te)
X_test_te = te_scaler.transform(X_test_te)

X_train_full = np.hstack([X_train_num, X_train_te])
X_test_full = np.hstack([X_test_num, X_test_te])
print(f"Full Train shape: {X_train_full.shape}, Full Test shape: {X_test_full.shape}")


In [ ]:

# 4. Neural Network Architecture
class TEMLP(nn.Module):
    def __init__(self, input_dim, out_sz):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, out_sz)
        )
    def forward(self, x):
        return self.net(x)


In [ ]:

# 5. Training Loop with Inference
epochs = 20
batch_size = 1024
test_tensor = torch.tensor(X_test_full, dtype=torch.float32).to(device)
test_probs = np.zeros((len(test_df), 3))
oof_preds = np.zeros(len(train_df))

class_counts = np.bincount(y_train)
weights = torch.tensor(1. / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train)):
    print(f"\n--- Fold {fold+1} ---")
    model = TEMLP(X_train_full.shape[1], 3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    X_tr, y_tr = torch.tensor(X_train_full[train_idx], dtype=torch.float32), torch.tensor(y_train[train_idx], dtype=torch.long)
    X_va, y_va = torch.tensor(X_train_full[val_idx], dtype=torch.float32), torch.tensor(y_train[val_idx], dtype=torch.long)
    
    train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size, shuffle=False)
    
    best_acc = 0
    best_weights = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for bx, by in train_dl:
            bx, by = bx.to(device), by.to(device)
            out = model(bx)
            loss = criterion(out, by)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        scheduler.step()
        
        # Validation
        model.eval()
        val_preds = []
        with torch.no_grad():
            for bx, _ in val_dl:
                out = model(bx.to(device))
                val_preds.append(out.cpu().numpy())
        val_preds = np.vstack(val_preds)
        val_acc = balanced_accuracy_score(y_va.numpy(), val_preds.argmax(axis=1))
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = model.state_dict()
            oof_preds[val_idx] = val_preds.argmax(axis=1)
            
    print(f"Fold {fold+1} Best Balanced Accuracy: {best_acc:.4f}")
    scores.append(best_acc)
    
    # Load best weights for test inference
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        out = model(test_tensor)
        test_probs += torch.softmax(out, dim=1).cpu().numpy() / n_splits

print(f"\nOverall CV Balanced Accuracy: {np.mean(scores):.4f}")


In [ ]:

# 6. Submission Export
final_test_preds = test_probs.argmax(axis=1)
sub_df['health_condition'] = le_target.inverse_transform(final_test_preds)
sub_df.to_csv('submission.csv', index=False)
print("Saved submission to submission.csv")
sub_df.head()
